# Metaclasses

## Modifying Methods

Write a metaclass `LogComputeMeta` that transforms all the regular methods of a class (neither `@staticmethod` nor `@classmethod`) whose name contains the word "compute" with the `log_calls` decorator of the next cell, which logs calls to the methods it decorates. An attribute that isn't a method, like `compute_mode` below, doesn't change.

With this class:

```python
class A(metaclass=LogComputeMeta):
  compute_mode = "exact"

  def compute_addition(self, a, b):
    return a + b


a = A()
a.compute_addition(1, 2)
print(a.compute_mode)
```

You should get:

```text
compute_addition called with args (1, 2) and kwargs {} computed 3
exact
```

**Useful documentation**

- [`inspect.isfunction`](https://docs.python.org/3/library/inspect.html#inspect.isfunction)

In [ ]:
import functools


def log_calls(f):

  @functools.wraps(f)
  def wrapper(self, *args, **kwargs):
    result = f(self, *args, **kwargs)
    print(f"{f.__name__} called with args {args} and kwargs {kwargs} computed {result}")
    return result

  return wrapper

In [ ]:
# Your code here

### Additional hints

`__new__` receives in `namespace` the dictionary of the names defined in the class body. Build a new one, where the targeted methods go through `log_calls`, and give it to `super().__new__`. `inspect.isfunction(v)` is `True` for a regular method and `False` for `compute_mode`'s string.

```python
import inspect


class LogComputeMeta(type):
  def __new__(mcs, name, bases, namespace):
    new_namespace = {k: (log_calls(v) if ... else v) for k, v in namespace.items()}
    return super().__new__(mcs, name, bases, new_namespace)
```

### Solution

In [ ]:
import inspect


class LogComputeMeta(type):
  def __new__(mcs, name, bases, namespace):
    # In the namespace, staticmethod and classmethod aren't functions: isfunction
    # leaves them aside, like compute_mode. With callable, a staticmethod, callable
    # since Python 3.10, would go through log_calls and receive an extra self
    new_namespace = {
      k: (log_calls(v) if "compute" in k and inspect.isfunction(v) else v)
      for k, v in namespace.items()
    }
    return super().__new__(mcs, name, bases, new_namespace)


class A(metaclass=LogComputeMeta):
  compute_mode = "exact"

  def compute_addition(self, a, b):
    return a + b


a = A()
a.compute_addition(1, 2)
# compute_mode isn't a function, so it stays a string
print(a.compute_mode)

## Validating Subclasses: Metaclass or `__init_subclass__`

We want every subclass of `Exporter` to define an `extension` class attribute, a string starting with a dot, and a `TypeError` to be raised as soon as the class is defined otherwise.

1. Implement this check with an `ExporterMeta` metaclass.
2. Then implement it without a metaclass, with `__init_subclass__`.

In both cases, `class CsvExporter(Exporter): extension = ".csv"` must be accepted, and `class BrokenExporter(Exporter): pass` must raise a `TypeError`.

**Useful documentation**

- [Metaclasses](https://docs.python.org/3/reference/datamodel.html#metaclasses)

In [ ]:
# Your code here

### Additional hints

A metaclass's `__init__` receives the class already created: first call `super().__init__(name, bases, namespace)`, then check. `if bases` leaves out `Exporter` itself, which has no parents; `__init_subclass__` doesn't need it, since it's only called for subclasses. `getattr(cls, "extension", None)` reads the attribute, inherited or not, with no error when it's missing, and `isinstance(extension, str)` checks that it's a string before calling `startswith`.

To check: `extension = ".csv"` passes; a class with no `extension`, `extension = "csv"` and `extension = 1` raise a `TypeError`.

```python
class ExporterMeta(type):
  def __init__(cls, name, bases, namespace):
    super().__init__(name, bases, namespace)
    extension = getattr(cls, "extension", None)
    if bases and not ...:
      raise TypeError(f"{name} must define an extension starting with a dot")


class Exporter(metaclass=ExporterMeta):
  pass


# Without a metaclass
class Exporter:
  def __init_subclass__(cls, **kwargs):
    super().__init_subclass__(**kwargs)
    extension = getattr(cls, "extension", None)
    ...
```

### Solution

In [ ]:
import abc


class ExporterMeta(type):
  def __init__(cls, name, bases, namespace):
    super().__init__(name, bases, namespace)
    extension = getattr(cls, "extension", None)
    if bases and not (isinstance(extension, str) and extension.startswith(".")):
      raise TypeError(f"{name} must define an extension starting with a dot")


class Exporter(metaclass=ExporterMeta):
  pass


class CsvExporter(Exporter):
  extension = ".csv"


try:

  class BrokenExporter(Exporter):
    pass

except TypeError as error:
  print(error)

try:

  class AbstractExporter(Exporter, abc.ABC):
    extension = ".txt"

except TypeError as error:
  print(error)


# Without a metaclass
class Exporter:
  def __init_subclass__(cls, **kwargs):
    super().__init_subclass__(**kwargs)
    extension = getattr(cls, "extension", None)
    if not (isinstance(extension, str) and extension.startswith(".")):
      raise TypeError(f"{cls.__name__} must define an extension starting with a dot")


class CsvExporter(Exporter):
  extension = ".csv"


try:

  class BrokenExporter(Exporter):
    pass

except TypeError as error:
  print(error)


class AbstractExporter(Exporter, abc.ABC):
  extension = ".txt"


print(type(AbstractExporter))

`__init_subclass__` gives the same result without a metaclass: the code is simpler, and the class can still be combined with other classes that have their own metaclass. `AbstractExporter`, which also inherits from `abc.ABC`, causes a metaclass conflict with `ExporterMeta`, and passes with `__init_subclass__`.

## Singleton Pattern Implementation

The singleton pattern involves always returning the same instance when client code requests a new instance of a class.

Create a metaclass `SingletonMeta` that implements this pattern. Note that a metaclass can be applied to multiple classes: each one has its own instance.

A metaclass can override several methods. Here, the simplest will be to redefine `__call__`.

**Useful documentation**

- [`object.__call__`](https://docs.python.org/3/reference/datamodel.html#object.__call__)

In [ ]:
# Your code here

### Additional hints

The metaclass's `__call__` receives `cls`, the class client code requests an instance of, and `super().__call__(*args, **kwargs)` creates that instance (it calls `cls`'s `__new__` and `__init__`). Keep the instances in a dictionary whose keys are the classes.

To check with two classes `A` and `B`: `A() is A()` is `True`, `A() is B()` is `False`, and `A`'s `__init__` runs only once.

```python
class SingletonMeta(type):
  _instances = {}

  def __call__(cls, *args, **kwargs):
    if cls not in SingletonMeta._instances:
      SingletonMeta._instances[cls] = ...
    return ...
```

### Solution

In [ ]:
class SingletonMeta(type):
  _instances = {}

  def __call__(cls, *args, **kwargs):
    if cls not in SingletonMeta._instances:
      SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
    return SingletonMeta._instances[cls]


class A(metaclass=SingletonMeta):
  def __init__(self):
    print("Creating the instance of A")


class B(metaclass=SingletonMeta):
  pass


print(A() is A())
print(A() is B())

Three limits to know before using this solution:

- **A module is often enough.** Python runs a module only once: an object created at module level is already unique, without a metaclass.
- **`__new__` instead of `__call__`.** A singleton written with the class's `__new__` runs `__init__` again on every call, because `type.__call__` calls `__init__` on the object `__new__` returns. With the metaclass's `__call__`, `__init__` runs only once.
- **Threads.** Several threads can pass the `cls not in ...` test before the first instance exists, and each one creates its own. The next cell shows it with four threads and a slow `__init__`, then fixes it with a `threading.Lock`.

In [ ]:
import threading
import time


def count_instances(target):
  objects = []
  threads = [
    threading.Thread(target=lambda: objects.append(target())) for _ in range(4)
  ]
  for thread in threads:
    thread.start()
  for thread in threads:
    thread.join()
  return len({id(obj) for obj in objects})


class Slow(metaclass=SingletonMeta):
  def __init__(self):
    time.sleep(0.1)


print(count_instances(Slow))


class SingletonLockMeta(type):
  _instances = {}
  _lock = threading.Lock()

  def __call__(cls, *args, **kwargs):
    with SingletonLockMeta._lock:
      if cls not in SingletonLockMeta._instances:
        SingletonLockMeta._instances[cls] = super().__call__(*args, **kwargs)
    return SingletonLockMeta._instances[cls]


class SlowLocked(metaclass=SingletonLockMeta):
  def __init__(self):
    time.sleep(0.1)


print(count_instances(SlowLocked))

## Class Decorator or Metaclass

A class decorator is often enough instead of a metaclass. Compare the two.

1. Write the class decorator `log_compute`, which does the same job as `LogComputeMeta` (first exercise) on the class it decorates.
2. Apply it to an `Addition` class with a `compute_addition` method, then write a `Product` subclass that adds a `compute_product` method.
3. Write the same classes with the metaclass: `AdditionMeta`, with `metaclass=LogComputeMeta`, and its subclass `ProductMeta`.
4. Call `compute_product` on an instance of `Product` and on an instance of `ProductMeta`. Which call is logged, and why?

In [ ]:
# Your code here

### Additional hints

A class decorator receives the class and returns it. `vars(cls)` gives its attributes, and `setattr(cls, name, value)` replaces one. Loop over a copy, `list(vars(cls).items())`, rather than over the dictionary `setattr` modifies.

```python
import inspect


def log_compute(cls):
  for name, value in list(vars(cls).items()):
    if ...:
      setattr(cls, name, ...)
  return cls
```

### Solution

In [ ]:
import inspect


def log_compute(cls):
  for name, value in list(vars(cls).items()):
    if "compute" in name and inspect.isfunction(value):
      setattr(cls, name, log_calls(value))
  return cls


@log_compute
class Addition:
  def compute_addition(self, a, b):
    return a + b


class Product(Addition):
  def compute_product(self, a, b):
    return a * b


class AdditionMeta(metaclass=LogComputeMeta):
  def compute_addition(self, a, b):
    return a + b


class ProductMeta(AdditionMeta):
  def compute_product(self, a, b):
    return a * b


print("Decorator:")
Product().compute_addition(1, 2)
Product().compute_product(2, 3)
print("Metaclass:")
ProductMeta().compute_addition(1, 2)
ProductMeta().compute_product(2, 3)

Only the metaclass logs `compute_product`. The decorator applies once, to the class it decorates: `Product` inherits the already decorated `compute_addition`, but its new method doesn't go through it. A metaclass is inherited: `type(ProductMeta)` is `LogComputeMeta`, whose `__new__` also runs for subclasses. `__init_subclass__`, which is inherited too, would give this behavior without a metaclass.